# D1-S1-Nb1: Step-by-step Build of our Workcell

Showcase: how to take your first steps in PLR

## Set your `protocol_mode`

...so it is easy to switch between simulation and execution

In [ ]:
star_simulation = True  # True or False

run_identifier = "nb1"

## Library Import

In [ ]:
import datetime
import logging
import time


---
# Create a Facility resource
...so everything that follows has one highest-level reference frame

In [ ]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

In [ ]:
facility = Facility(name="facility", size_x=2_400, size_y=1_000, size_z=0)
facility

In [ ]:
viewer = Viewer3D(
    root = facility,
    name="nb1_first_steps"
    )

await viewer.start()

## Ready the STAR(let)

N.B>.: we're using `STARlet` because its smalle size makes this workshop (where we're all on small-screen laptops) easier.
To switch to a STAR in your own lab simply delete the last 3 letters:
`STARlet` -> `STAR`

In [ ]:
from pylabrobot.hamilton.star import STARlet

star = STARlet(
    simulation=star_simulation,
    )

In [ ]:
facility.assign_child_resource(
    star,
    location=Coordinate(x=800.0, y=0.0, z=0.0)
)

## Run `.setup()` for the Features to be loaded
...from a config file

In [ ]:
await star.setup()

## Same for Prep

In [ ]:
prep_simulation = True  # True or False

In [ ]:
from pylabrobot.hamilton.prep import Prep

prep = Prep(
    simulation=prep_simulation,
    )

facility.assign_child_resource(
    prep,
    location=Coordinate(x=0.0, y=0.0, z=0.0)
)

In [ ]:
await prep.setup()
await prep.lights.turn_on()

---
# Configuration JSON
...what the device *is*, as a file

At `setup()`, `discover()` asks the device what is fitted: channels, heads, iSWAP, autoload, firmware.
The answer is kept on `.driver.configuration`, and `save_configuration()` writes it to a JSON file.

That one file is used two ways:
1. **Simulation:** a simulated device answers exactly as the file says, no robot needed.
2. **Real device:** passed as `declared_configuration_json=`, it is checked against what the device answers, and `setup()` stops if they differ.

## What `setup()` read

In [ ]:
for device in (star, prep):
    c = device.driver.configuration
    print(device.name, "|", c.serial_number, "|", c.firmware_version)

In [ ]:
c = star.driver.configuration
{"channels": c.num_pip_channels, "96-head": c.head96_installed, "iSWAP": c.kb_iswap_installed, "autoload": c.autoload_installed}

In [ ]:
p = prep.driver.configuration
{"channels": p.num_channels, "8-channel head": p.head8_installed, "deck sites": len(p.deck_sites)}

## Where a simulation's answers come from

In [ ]:
import os

print(os.path.basename(star.driver.declared_configuration_json or "none declared"))
print(os.path.basename(prep.driver.declared_configuration_json or "none declared"))

## Save it

On a physical device, you are likely interested to
1. perform a discovery of your device's configuration 
2. save it
3. set your simulation mode to always use your configuration json, so simulation is tailored to your device

In [ ]:
import json

os.makedirs("_logs", exist_ok=True)
star_json = f"_logs/{run_identifier}_star_configuration.json"
star.driver.save_configuration(star_json)

with open(star_json) as f:
    saved = json.load(f)

{section: len(fields) for section, fields in saved.items()}  # fields per section

In [ ]:
saved["device"]["num_pip_channels"], saved["device"]["head96_installed"]

## Edit the file, get a different device
The same file with a few values changed: a STARlet with 4 channels, no 96-head and no autoload, and without the extension housing.

In [ ]:
saved["device"]["num_pip_channels"] = 4

saved["device"]["head96_installed"] = False  # the device, and the arm that carries it
saved["device"]["left_arm"]["head96_installed"] = False
saved["arms"]["left"].pop("head96", None)

saved["device"]["autoload_installed"] = False
saved.pop("autoload", None)

star_4ch_json = f"{run_identifier}_star_4ch_configuration.json"  # next to this notebook
with open(star_4ch_json, "w") as f:
    json.dump(saved, f, indent=2)

star_4ch = STARlet(
    simulation=True,
    declared_configuration_json=star_4ch_json,
    name="star_4ch",
    extension_housing=False,  # a resource, not in the file
)
await star_4ch.setup()

facility.assign_child_resource(
    star_4ch,
    location=Coordinate(x=800.0, y=-1_100.0, z=0.0)
)

{
    "channels": star_4ch.pipettes.num_channels,
    "96-head": star_4ch.head96 is not None,
    "autoload": star_4ch.autoload is not None,
}

In [ ]:
await star_4ch.stop()

## On the real device
Save the configuration once from your robot, keep the file next to your protocol, and declare it:

```python
star = STARlet(declared_configuration_json="our_star.json")
await star.setup()  # stops if the STAR answers differently
```

`setup()` compares what is fitted: channels, heads, iSWAP, autoload. Serial number and firmware are not compared,
so one file serves every STARlet of the same build.

Declare the file we just edited on our 8-channel STARlet, and setup stops before anything moves:

```
ValueError: the declared configuration does not describe this device:
  num_pip_channels: declared 4, device answers 8
  head96_installed: declared False, device answers True
  autoload_installed: declared False, device answers True
  left_arm.head96_installed: declared False, device answers True
```

---
# Shutdown

In [ ]:
await star.stop()

In [ ]:
await prep.stop()

In [ ]:
await viewer.stop()